# QMCF-Net: Full Working Implementation (PyTorch + PennyLane)

Implements a runnable hybrid multimodal regression pipeline based on the supplied methodology: audience consensus targets (SEN/TRU/ENG), participant disagreement, four modality inputs (text/image/emotion/context), 16-qubit differentiable circuit, uncertainty-aware loss, train/validation/test split, metrics, modality ablation, and resource reporting.

**Important data note:** the methodology describes 800 stimuli and participant-level ratings, but those datasets are not attached here. This notebook therefore expects your own prepared files. It will not invent real audience ratings or claim results from the paper. A compact feature-table interface is used so training is reproducible; raw OCR/transformer/image-feature extraction can be added before this notebook.

## 1. Install dependencies

Run this cell in Google Colab. PennyLane's default.qubit simulator is a differentiable state-vector simulator. A 16-qubit state has 65,536 amplitudes, so use a GPU for the classical front-end if available and begin with a small batch size.

In [ ]:
%pip -q install pennylane torch torchvision scikit-learn pandas numpy matplotlib seaborn joblib

## 2. Mount Google Drive and configure paths

Prepare:
- `stimuli.csv`: one row per stimulus with `stimulus_id`, `domain` (`AD` or `SM`), and modality feature columns.
- `ratings.csv`: one row per valid participant-stimulus rating, with `stimulus_id`, `SEN1`, `TRU1`–`TRU3`, and `ENG1`–`ENG3`. All seven items must be present for a rating.
- Four feature blocks in `stimuli.csv`: `txt_0...`, `img_0...`, `emo_0...`, and `ctx_0...`. Feature widths may vary; the model projects them to 64 latent dimensions. Recommended widths are text 384, image 576, emotion 14, and context one-hot plus logo indicator.

The notebook also supports a clearly marked synthetic smoke test if the CSVs are unavailable; do not use synthetic results as scientific findings.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, random, json, time, hashlib, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# CHANGE THESE PATHS
DATA_DIR = '/content/drive/MyDrive/QMCF-Net'
STIMULI_CSV = os.path.join(DATA_DIR, 'stimuli.csv')
RATINGS_CSV = os.path.join(DATA_DIR, 'ratings.csv')
OUTPUT_DIR = os.path.join(DATA_DIR, 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Device:', DEVICE)

## 3. Load stimuli and construct participant-derived consensus targets

For each stimulus, the code retains only rows with all seven rating items. SEN is `SEN1`; TRU and ENG are item means; sample standard deviation and valid rating count are computed per construct. These labels are kept separate from model inputs.

In [ ]:
def load_and_aggregate(stimuli_path, ratings_path):
    stimuli = pd.read_csv(stimuli_path)
    ratings = pd.read_csv(ratings_path)
    required_stim = {'stimulus_id', 'domain'}
    required_rate = {'stimulus_id','SEN1','TRU1','TRU2','TRU3','ENG1','ENG2','ENG3'}
    if not required_stim.issubset(stimuli.columns):
        raise ValueError(f'stimuli.csv must contain {sorted(required_stim)}')
    if not required_rate.issubset(ratings.columns):
        raise ValueError(f'ratings.csv must contain {sorted(required_rate)}')
    item_cols = ['SEN1','TRU1','TRU2','TRU3','ENG1','ENG2','ENG3']
    ratings = ratings.dropna(subset=item_cols).copy()
    ratings['SEN'] = ratings['SEN1'].astype(float)
    ratings['TRU'] = ratings[['TRU1','TRU2','TRU3']].astype(float).mean(axis=1)
    ratings['ENG'] = ratings[['ENG1','ENG2','ENG3']].astype(float).mean(axis=1)
    target_cols = ['SEN','TRU','ENG']
    agg_mean = ratings.groupby('stimulus_id')[target_cols].mean().add_prefix('mu_')
    agg_std = ratings.groupby('stimulus_id')[target_cols].std(ddof=1).add_prefix('sigma_')
    agg_n = ratings.groupby('stimulus_id').size().rename('n_ratings')
    data = stimuli.merge(agg_mean, on='stimulus_id', how='inner')
    data = data.merge(agg_std, on='stimulus_id', how='left').merge(agg_n, on='stimulus_id', how='left')
    data = data.dropna(subset=['mu_SEN','mu_TRU','mu_ENG'])
    data[['sigma_SEN','sigma_TRU','sigma_ENG']] = data[['sigma_SEN','sigma_TRU','sigma_ENG']].fillna(0.0)
    return data, ratings

if os.path.exists(STIMULI_CSV) and os.path.exists(RATINGS_CSV):
    data, ratings_valid = load_and_aggregate(STIMULI_CSV, RATINGS_CSV)
    print('Loaded stimuli:', len(data), '| valid ratings:', len(ratings_valid))
    print(data[['mu_SEN','mu_TRU','mu_ENG','n_ratings']].describe())
else:
    print('Data files not found. Run the synthetic smoke-test cell in Section 4 to verify the code, or upload your real CSVs.')
    data = None

## 4. Feature selection and smoke-test data

Feature columns must use the prefixes below. For a real run, all modality features must be computed from content only, without using rating targets, participant variables, domain, or source dataset as features. In this compact implementation, the pretrained encoders are expected to have been run before training and their outputs stored in the CSV.

In [ ]:
MODALITIES = ['txt','img','emo','ctx']
def get_feature_columns(df):
    cols = {m: [c for c in df.columns if c.startswith(m + '_') and c[len(m)+1:].isdigit()] for m in MODALITIES}
    missing = [m for m in MODALITIES if not cols[m]]
    if missing:
        raise ValueError('Missing feature columns for: ' + ', '.join(missing) +
                         '. Add numeric columns such as txt_0, img_0, emo_0, ctx_0.')
    for m in MODALITIES:
        cols[m] = sorted(cols[m], key=lambda c: int(c.split('_')[-1]))
    return cols

if data is None:
    # Synthetic smoke test only. Replace with real CSVs for research.
    n=120
    rng=np.random.default_rng(SEED)
    data=pd.DataFrame({'stimulus_id':[f'S{i:04d}' for i in range(n)],
                       'domain':np.array(['AD','SM']*(n//2))})
    for m,d in [('txt',32),('img',24),('emo',14),('ctx',8)]:
        arr=rng.normal(size=(n,d))
        for k in range(d): data[f'{m}_{k}']=arr[:,k]
    latent=rng.normal(size=(n,3))
    for j,q in enumerate(['SEN','TRU','ENG']):
        data[f'mu_{q}']=np.clip(4+latent[:,j]*.7,1.1,6.9)
        data[f'sigma_{q}']=rng.uniform(.4,1.2,n)
    data['n_ratings']=36
    print('WARNING: synthetic data generated for software testing only.')

feature_cols = get_feature_columns(data)
print({m:len(feature_cols[m]) for m in MODALITIES})

## 5. Split data and scale using training data only

This is a stimulus-disjoint holdout. If you have the original 25 rating-block IDs, replace the random split with the exact block-wise split described in the methodology and preserve participant disjointness.

In [ ]:
target_cols = ['mu_SEN','mu_TRU','mu_ENG']
sigma_cols = ['sigma_SEN','sigma_TRU','sigma_ENG']
indices = np.arange(len(data))
train_idx, test_idx = train_test_split(indices, test_size=0.20, random_state=SEED,
                                       stratify=data['domain'] if data['domain'].nunique()>1 else None)
train_idx, val_idx = train_test_split(train_idx, test_size=0.20, random_state=SEED,
                                      stratify=data.iloc[train_idx]['domain'] if data['domain'].nunique()>1 else None)
scalers = {}
X = {}
for m in MODALITIES:
    raw = data[feature_cols[m]].apply(pd.to_numeric, errors='coerce').fillna(0).to_numpy(np.float32)
    mean = raw[train_idx].mean(axis=0, keepdims=True)
    std = raw[train_idx].std(axis=0, keepdims=True)
    std[std < 1e-3] = 1.0
    X[m] = (raw-mean)/std
    scalers[m] = {'mean':mean, 'std':std, 'columns':feature_cols[m]}
Y = data[target_cols].to_numpy(np.float32)
SIGMA = data[sigma_cols].to_numpy(np.float32)
N_RATINGS = data['n_ratings'].fillna(1).to_numpy(np.float32).reshape(-1,1)
print('Train/val/test:', len(train_idx), len(val_idx), len(test_idx))

## 6. QMCF-Net model

The differentiable 16-qubit state-vector circuit includes:
- **QSPE-like encoding:** modality-specific direction/intensity projections and learned allocation.
- **MQIL-like interaction:** two variational layers with within-modality and cross-modality ZZ entanglement, with interaction strength routed by learned reliability/compatibility.
- **TSEM-like target structuring:** target-specific readout rotations and signed register coupling.
- Three 32-value readouts with bounded mean predictions in (1, 7) and positive predictive variance.

This is a runnable reference implementation of the method's computational flow. Some manuscript details marked `TO CONFIRM` (specific pretrained checkpoint revisions, face detector, topic taxonomy, learning-rate/epoch settings, and preregistration) require the authors' actual choices before claiming exact experimental reproduction.

In [ ]:
import pennylane as qml
from torch.utils.data import Dataset, DataLoader

N_QUBITS = 16
N_LAYERS = 2
N_TARGETS = 3
QDEV = qml.device('default.qubit', wires=N_QUBITS, shots=None)

def make_qnode():
    @qml.qnode(QDEV, interface='torch', diff_method='backprop')
    def circuit(angles, intra, cross, tsem_rot, tsem_couple, readout):
        # angles: [16,2]; single-qubit rotations
        for w in range(N_QUBITS):
            qml.RY(angles[w,0], wires=w)
            qml.RZ(angles[w,1], wires=w)
        # Two MQIL-like layers
        for layer in range(N_LAYERS):
            offset = layer * N_QUBITS * 2
            for w in range(N_QUBITS):
                qml.RY(intra['rot'][layer,w,0], wires=w)
                qml.RZ(intra['rot'][layer,w,1], wires=w)
            # nearest-neighbour entanglement within each 4-qubit modality block
            for m in range(4):
                for k in range(3):
                    a=m*4+k; b=a+1
                    qml.IsingZZ(intra['zz'][layer,m,k], wires=[a,b])
            # all six modality pairs, same position, reliability/compatibility-routed
            pairs=[(0,1),(0,2),(0,3),(1,2),(1,3),(2,3)]
            for pi,(m1,m2) in enumerate(pairs):
                for k in range(4):
                    qml.IsingZZ(cross[layer,pi,k], wires=[m1*4+k,m2*4+k])
        # Target registers are qubit positions 0,1,2; shared mediator position 3
        # Apply target-structured stimulus-conditioned rotations.
        for g in range(4):
            for m in range(4):
                w=m*4+g
                qml.RY(tsem_rot[g,m,0], wires=w)
                qml.RZ(tsem_rot[g,m,1], wires=w)
        reg_pairs=[(0,1),(0,2),(0,3),(1,2),(1,3),(2,3)]
        for pi,(g1,g2) in enumerate(reg_pairs):
            for m in range(4):
                qml.IsingZZ(tsem_couple[pi,m], wires=[m*4+g1,m*4+g2])
        for g in range(4):
            for m in range(4):
                qml.RY(readout[g,m], wires=m*4+g)
        # 12 target/mediator single Z expectations + 24 ZZ correlators per target
        outputs=[]
        for q in range(3):
            pos=q
            for m in range(4): outputs.append(qml.expval(qml.PauliZ(m*4+pos)))
            for m in range(4): outputs.append(qml.expval(qml.PauliZ(m*4+3)))
            # 24 correlations: six pairs within target/mediator registers + cross target/other registers
            pairs_q=[]
            tq=[m*4+q for m in range(4)]
            sh=[m*4+3 for m in range(4)]
            for a in range(4):
                for b in range(a+1,4):
                    pairs_q.append((tq[a],tq[b]))
                    pairs_q.append((sh[a],sh[b]))
            for a in range(4):
                for g in range(3):
                    if g != q: pairs_q.append((tq[a],a*4+g))
            for a,b in pairs_q[:24]:
                outputs.append(qml.expval(qml.PauliZ(a) @ qml.PauliZ(b)))
            # pad only if needed to guarantee exactly 32 values
            while len(outputs) % 32 != 0:
                outputs.append(qml.expval(qml.PauliZ(pos)))
        return tuple(outputs)

    return circuit

QNODE = make_qnode()

In [ ]:
class QMCFNet(nn.Module):
    def __init__(self, input_dims):
        super().__init__()
        self.modalities = MODALITIES
        self.lat = nn.ModuleDict()
        self.to_q = nn.ModuleDict()
        self.to_align = nn.ModuleDict()
        self.placeholder = nn.ParameterDict()
        self.alloc = nn.ModuleDict()
        self.dir_angle = nn.ModuleDict()
        self.int_angle = nn.ModuleDict()
        self.reliability = nn.ModuleDict()
        for m in MODALITIES:
            self.lat[m] = nn.Sequential(nn.Linear(input_dims[m],64), nn.GELU(), nn.LayerNorm(64))
            self.to_q[m] = nn.Linear(64,4)
            self.to_align[m] = nn.Linear(64,16)
            self.placeholder[m] = nn.Parameter(torch.zeros(64))
            self.alloc[m] = nn.Linear(8,4)
            self.dir_angle[m] = nn.Linear(4,4)
            self.int_angle[m] = nn.Linear(4,4)
            self.reliability[m] = nn.Sequential(nn.Linear(input_dims[m],16),nn.GELU(),nn.Linear(16,1),nn.Sigmoid())
        # Shared variational angles for MQIL
        self.mq_rot = nn.Parameter(torch.randn(N_LAYERS,16,2)*0.05)
        self.mq_zz = nn.Parameter(torch.randn(N_LAYERS,4,3)*0.05)
        self.mq_cross = nn.Parameter(torch.randn(N_LAYERS,6,4)*0.05)
        self.hyper = nn.Sequential(nn.Linear(10,16),nn.GELU())
        self.hyper_rot = nn.ModuleList([nn.Linear(16,32) for _ in range(N_LAYERS)])
        self.hyper_ib = nn.ModuleList([nn.Linear(16,12) for _ in range(N_LAYERS)])
        self.hyper_cm = nn.ModuleList([nn.Linear(16,24) for _ in range(N_LAYERS)])
        self.tsem_context = nn.Sequential(nn.Linear(26,16),nn.GELU(),nn.Linear(16,16),nn.LayerNorm(16))
        self.tsem_rot = nn.ModuleList([nn.Linear(16,8) for _ in range(4)])
        self.tsem_couple = nn.ModuleList([nn.Linear(16,4) for _ in range(6)])
        self.readout = nn.ModuleList([nn.Linear(16,4) for _ in range(4)])
        self.mean_heads = nn.ModuleList([nn.Linear(32,1) for _ in range(3)])
        self.var_heads = nn.ModuleList([nn.Linear(32,1) for _ in range(3)])
        self.qnode = QNODE

    def forward(self, inputs, return_aux=False):
        batch = inputs['txt'].shape[0]
        z, u, a, rho = {}, {}, {}, {}
        for m in self.modalities:
            h = self.lat[m](inputs[m])
            z[m] = h
            u[m] = self.to_q[m](h)
            a[m] = F.normalize(self.to_align[m](h), dim=-1, eps=1e-6)
            rho[m] = self.reliability[m](inputs[m]).squeeze(-1)
        # Batch simulation one stimulus at a time: correct but intentionally conservative for memory.
        pred_all=[]; var_all=[]; measurements_all=[]
        for i in range(batch):
            ang=[]
            for m in self.modalities:
                ui=u[m][i]
                p=ui/(torch.linalg.vector_norm(ui)+1e-6)
                kappa=torch.tanh(torch.log1p(torch.linalg.vector_norm(ui)))
                beta=torch.sigmoid(self.alloc[m](torch.cat([p,kappa.repeat(4)])))
                ay=torch.pi*torch.tanh(self.dir_angle[m](beta*p))
                az=torch.pi*torch.tanh(self.int_angle[m]((1-beta)*kappa))
                ang.extend([torch.stack([ay[k],az[k]]) for k in range(4)])
            angles=torch.stack(ang)
            # pairwise compatibilities and routed cross-gate strengths
            pair_ids=[(0,1),(0,2),(0,3),(1,2),(1,3),(2,3)]
            compat=[]; route=[]
            for pidx,(m1,m2) in enumerate(pair_ids):
                if m1<3 and m2<3:
                    c=(1+(a[self.modalities[m1]][i]*a[self.modalities[m2]][i]).sum())/2
                else: c=torch.tensor(0.5,device=angles.device)
                compat.append(c)
                r1=rho[self.modalities[m1]][i]; r2=rho[self.modalities[m2]][i]
                route.append(r1*r2*torch.sigmoid(torch.stack([r1,r2,c,r1*r2]).sum()))
            cond=torch.cat([torch.stack([rho[m][i] for m in self.modalities]),torch.stack(compat)])
            hidden=self.hyper(cond)
            rot=[]; ib=[]; cm=[]
            for layer in range(N_LAYERS):
                rot.append(self.mq_rot[layer] + (torch.pi/4)*torch.tanh(self.hyper_rot[layer](hidden)).reshape(16,2))
                ib.append(self.mq_zz[layer] + (torch.pi/4)*torch.tanh(self.hyper_ib[layer](hidden)).reshape(4,3))
                cm.append((self.mq_cross[layer] + (torch.pi/4)*torch.tanh(self.hyper_cm[layer](hidden)).reshape(6,4))*torch.stack(route).unsqueeze(-1))
            intra={'rot':torch.stack(rot),'zz':torch.stack(ib)}
            cross=torch.stack(cm)
            context=torch.cat([cond,*[u[m][i] for m in self.modalities]])
            o=self.tsem_context(context)
            tsrot=torch.stack([torch.pi*torch.tanh(self.tsem_rot[g](o)).reshape(4,2) for g in range(4)])
            tscpl=torch.stack([torch.tanh(self.tsem_couple[p](o)) for p in range(6)])
            rd=torch.stack([torch.pi*torch.tanh(self.readout[g](o)) for g in range(4)])
            measurements=self.qnode(angles,intra,cross,tsrot,tscpl,rd)
            e=torch.stack([v if torch.is_tensor(v) else torch.tensor(v,device=angles.device) for v in measurements]).to(torch.float32)
            e=e.reshape(3,32)
            means=[]; vars_=[]
            for q in range(3):
                means.append(1+6*torch.sigmoid(self.mean_heads[q](e[q])).squeeze())
                vars_.append(F.softplus(self.var_heads[q](e[q])).squeeze()+1e-6)
            pred_all.append(torch.stack(means)); var_all.append(torch.stack(vars_)); measurements_all.append(e)
        pred=torch.stack(pred_all); var=torch.stack(var_all); measurements=torch.stack(measurements_all)
        if return_aux:
            return pred,var, {'u':u,'a':a,'rho':rho,'measurements':measurements}
        return pred,var

model = QMCFNet({m:len(feature_cols[m]) for m in MODALITIES}).to(DEVICE)
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))

## 7. Training loop and uncertainty-aware objective

The loss follows the Gaussian negative log-likelihood form in Eq. (47), using participant disagreement and rating count only in the loss. Geometry/alignment/dependency auxiliary terms are optional toggles here; the implementation includes a stable multi-target dependency penalty. For a publication-grade exact reproduction, implement and validate the pairwise QSPE geometry and contrastive alignment terms exactly as defined in Eqs. (23) and (25).

In [ ]:
class FeatureDataset(Dataset):
    def __init__(self, ids):
        self.ids=np.asarray(ids)
    def __len__(self): return len(self.ids)
    def __getitem__(self, ix):
        i=int(self.ids[ix])
        inp={m:torch.tensor(X[m][i],dtype=torch.float32) for m in MODALITIES}
        return inp, torch.tensor(Y[i]), torch.tensor(SIGMA[i]), torch.tensor(N_RATINGS[i,0])

def collate_batch(batch):
    inp={m:torch.stack([b[0][m] for b in batch]) for m in MODALITIES}
    return inp, torch.stack([b[1] for b in batch]), torch.stack([b[2] for b in batch]), torch.stack([b[3] for b in batch])

BATCH_SIZE=4  # 16-qubit differentiable simulation; increase only after checking memory/speed
train_loader=DataLoader(FeatureDataset(train_idx),batch_size=BATCH_SIZE,shuffle=True,collate_fn=collate_batch)
val_loader=DataLoader(FeatureDataset(val_idx),batch_size=1,shuffle=False,collate_fn=collate_batch)
test_loader=DataLoader(FeatureDataset(test_idx),batch_size=1,shuffle=False,collate_fn=collate_batch)

def dependency_loss(pred, target):
    if pred.shape[0] < 3: return pred.new_tensor(0.0)
    terms=[]
    for a in range(3):
        for b in range(a+1,3):
            if pred[:,a].std(unbiased=False)>1e-3 and pred[:,b].std(unbiased=False)>1e-3 and target[:,a].std(unbiased=False)>1e-3 and target[:,b].std(unbiased=False)>1e-3:
                cp=torch.corrcoef(torch.stack([pred[:,a],pred[:,b]]))[0,1]
                ct=torch.corrcoef(torch.stack([target[:,a],target[:,b]]))[0,1]
                terms.append((cp-ct).square())
    return torch.stack(terms).mean() if terms else pred.new_tensor(0.0)

def objective(pred, var, y, sigma, n, lambda_dep=0.1):
    sampling_var=sigma.square()/n.clamp_min(1).unsqueeze(-1)
    total_var=(var+sampling_var).clamp_min(1e-6)
    nll=((y-pred).square()/(2*total_var)+0.5*torch.log(total_var)).mean()
    return nll + lambda_dep*dependency_loss(pred,y)

@torch.no_grad()
def evaluate_loss(model, loader):
    model.eval(); vals=[]
    for inp,y,sig,n in loader:
        inp={k:v.to(DEVICE) for k,v in inp.items()}
        y,sig,n=y.to(DEVICE),sig.to(DEVICE),n.to(DEVICE)
        p,v=model(inp)
        vals.append(objective(p,v,y,sig,n).item())
    return float(np.mean(vals))

EPOCHS=30
LR=1e-3
PATIENCE=6
optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
best_val=float('inf'); best_state=None; wait=0
history={'train_loss':[],'val_loss':[]}
for epoch in range(EPOCHS):
    model.train(); losses=[]
    for inp,y,sig,n in train_loader:
        inp={k:v.to(DEVICE) for k,v in inp.items()}
        y,sig,n=y.to(DEVICE),sig.to(DEVICE),n.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        pred,var=model(inp)
        loss=objective(pred,var,y,sig,n)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        optimizer.step()
        losses.append(float(loss.detach().cpu()))
    tr=float(np.mean(losses)); vl=evaluate_loss(model,val_loader)
    history['train_loss'].append(tr); history['val_loss'].append(vl)
    if vl<best_val:
        best_val=vl; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; wait=0
    else:
        wait+=1
    if wait>=PATIENCE: break
    print(f'Epoch {epoch+1:03d} | train={tr:.4f} | val={vl:.4f}')
if best_state is not None: model.load_state_dict(best_state)
print('Best validation loss:',best_val)

## 8. Test evaluation and save artifacts

Metrics are computed only on the held-out test stimuli. The test set should be evaluated once after all hyperparameter decisions are fixed.

In [ ]:
@torch.no_grad()
def predict_loader(model, loader):
    model.eval(); pp=[]; vv=[]; yy=[]; ss=[]; nnr=[]
    for inp,y,sig,n in loader:
        inp={k:v.to(DEVICE) for k,v in inp.items()}
        p,v=model(inp)
        pp.append(p.cpu().numpy()); vv.append(v.cpu().numpy())
        yy.append(y.numpy()); ss.append(sig.numpy()); nnr.append(n.numpy())
    return np.concatenate(pp),np.concatenate(vv),np.concatenate(yy),np.concatenate(ss),np.asarray(nnr).reshape(-1)

pred,var,ytrue,sigma_true,n_ratings=predict_loader(model,test_loader)
metric_rows=[]
for q,idx in zip(['SEN','TRU','ENG'],range(3)):
    err=ytrue[:,idx]-pred[:,idx]
    total_var=np.maximum(var[:,idx]+sigma_true[:,idx]**2/np.maximum(n_ratings,1),1e-6)
    metric_rows.append({'Target':q,'MAE':mean_absolute_error(ytrue[:,idx],pred[:,idx]),
                        'RMSE':mean_squared_error(ytrue[:,idx],pred[:,idx])**0.5,
                        'R2':r2_score(ytrue[:,idx],pred[:,idx]),
                        'Gaussian_NLL':float(np.mean(err**2/(2*total_var)+0.5*np.log(total_var))),
                        'Coverage_approx_95':float(np.mean(np.abs(err)<=1.96*np.sqrt(total_var)))})
metrics_df=pd.DataFrame(metric_rows)
display(metrics_df)
metrics_df.to_csv(os.path.join(OUTPUT_DIR,'test_metrics.csv'),index=False)
pred_df=pd.DataFrame({'stimulus_id':data.iloc[test_idx]['stimulus_id'].values})
for j,q in enumerate(['SEN','TRU','ENG']):
    pred_df[f'True_{q}']=ytrue[:,j]
    pred_df[f'Pred_{q}']=pred[:,j]
    pred_df[f'PredictiveVariance_{q}']=var[:,j]
pred_df.to_csv(os.path.join(OUTPUT_DIR,'test_predictions.csv'),index=False)
torch.save({'model_state_dict':model.state_dict(),'input_dims':{m:len(feature_cols[m]) for m in MODALITIES},
            'feature_cols':feature_cols,'seed':SEED,'best_val_loss':best_val},os.path.join(OUTPUT_DIR,'qmcf_net.pt'))
with open(os.path.join(OUTPUT_DIR,'feature_scalers.npz'),'wb') as f:
    np.savez(f,**{f'{m}_mean':scalers[m]['mean'] for m in MODALITIES},
             **{f'{m}_std':scalers[m]['std'] for m in MODALITIES})
print('Saved outputs to:',OUTPUT_DIR)

## 9. Training curve

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(history['train_loss'],label='Train loss')
plt.plot(history['val_loss'],label='Validation loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('QMCF-Net Training')
plt.legend(); plt.grid(alpha=.25); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'training_curve.png'),dpi=300)
plt.show()

## 10. Modality ablation (test set)

A modality is masked by replacing its feature vector with the training-set mean in standardized space (zero). This is a simple intervention proxy; the paper's Eq. (50) specifically masks the learned latent with the modality placeholder, so interpret this as a practical approximation.

In [ ]:
@torch.no_grad()
def predict_with_mask(model, loader, masked_modality=None):
    model.eval(); pp=[]
    for inp,y,sig,n in loader:
        inp={k:v.to(DEVICE) for k,v in inp.items()}
        if masked_modality is not None:
            inp[masked_modality]=torch.zeros_like(inp[masked_modality])
        p,v=model(inp); pp.append(p.cpu().numpy())
    return np.concatenate(pp)

base=predict_with_mask(model,test_loader)
abl_rows=[]
for m in MODALITIES:
    pm=predict_with_mask(model,test_loader,m)
    for q,j in zip(['SEN','TRU','ENG'],range(3)):
        abl_rows.append({'MaskedModality':m,'Target':q,'MeanAbsolutePredictionChange':float(np.mean(np.abs(base[:,j]-pm[:,j])))})
ablation_df=pd.DataFrame(abl_rows)
display(ablation_df)
ablation_df.to_csv(os.path.join(OUTPUT_DIR,'modality_ablation.csv'),index=False)

## 11. Resource and reproducibility report

In [ ]:
resource = {
    'qubits':16,
    'statevector_amplitudes':2**16,
    'mqil_layers':N_LAYERS,
    'measurement_settings_ideal_Z_observables':1,
    'trainable_parameters_total':sum(p.numel() for p in model.parameters() if p.requires_grad),
    'device':str(DEVICE),
    'seed':SEED,
    'n_stimuli_used':int(len(data)),
    'train_n':int(len(train_idx)),
    'validation_n':int(len(val_idx)),
    'test_n':int(len(test_idx)),
    'note':'State-vector simulation does not establish quantum computational advantage.'
}
with open(os.path.join(OUTPUT_DIR,'resource_report.json'),'w') as f: json.dump(resource,f,indent=2)
print(json.dumps(resource,indent=2))

## 12. Checklist before reporting scientific results

1. Replace the smoke-test data with real stimuli and participant ratings.
2. Use a frozen stimulus manifest and the exact block-wise split from the study (160 held-out test stimuli; five validation folds over the 640-stimulus development set).
3. Compute the raw-content features with documented model IDs/revisions, OCR configuration, face detector, and shared topic taxonomy. Do not include `domain`, source-dataset labels, participant attributes, or any rating-derived variable in the model input.
4. Ensure rating rows pass all quality checks and all seven items are present. Compute targets separately inside each partition if reproducing the partition-specific aggregation design.
5. Run the specified five seeds and compare against matched classical baselines and module ablations.
6. The manuscript's `TO CONFIRM` notes are unresolved. Record the final settings rather than silently assuming them.
7. Do not claim quantum advantage from a classical state-vector simulation.